# Neural CFR+ stability screen on the 18-claim spec

This notebook tests whether the long-run degradation we saw in neural CFR+ is caused by the averaging/training schedule rather than an unavoidable property of the current strategy process.

It compares:

- baseline optimized CFR+ (`linear` strategy weighting, constant LR);
- uniform strategy weighting;
- cosine LR decay;
- cosine LR decay plus uniform strategy weighting;
- cosine LR decay plus a periodic strategy replay reset.

Each variant is evaluated by exact BR against both:

- the learned average policy network;
- the current clipped-regret-derived policy.

Training time excludes exact evaluation time. Exact evaluations are intentionally periodic and can take non-trivial wall-clock time.

In [ ]:
from __future__ import annotations

import json
import math
import re
import time
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.deep_cfr import DeviceReservoirBuffer, ReservoirBuffer
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.policies.neural import compile_neural_to_dense
from liars_poker.serialization import save_policy


def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "liars_poker").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory.")


REPO_ROOT = find_repo_root()
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "cfr_plus_18_claim_stability_screen"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

print("repo:", REPO_ROOT)
print("torch:", torch.__version__)
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print("free / total GPU GiB:", round(free / 2**30, 2), "/", round(total / 2**30, 2))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=("RankHigh", "Pair", "TwoPair", "Trips"),
    suit_symmetry=True,
)

# This is intended to be long enough to see whether the earlier ~90 minute
# degradation is avoided, without becoming a multi-day screen.
RUN_MINUTES_PER_VARIANT = 100
EVAL_TARGET_MINUTES = [10, 20, 30, 45, 60, 75, 90, 100]
TRAVERSALS_PER_PLAYER = 1024
SEED = 7

DENSE_COMPILE_BATCH = 65_536
VALIDATION_RECORDS = 4096
SKIP_COMPLETED_VARIANTS = True

BASE_TRAINER_KWARGS = {
    "regret_hidden_sizes": (2048, 2048),
    "strategy_hidden_sizes": (512, 512),
    "device": device,
    "seed": SEED,
    "regret_buffer_capacity": 500_000,
    "strategy_buffer_capacity": 2_000_000,
    "learning_rate": 1e-3,
    "batch_size": 4096,
    "regret_train_steps": 24,
    "strategy_train_steps": 6,
    "strategy_weighting": "linear",
    "regret_positive_weight": 0.5,
    "validation_fraction": 0.02,
    "validation_buffer_capacity": 20_000,
    "traversal_backend": "gpu_native" if device.type == "cuda" else "recursive",
    "traversal_batch_size": 1024,
    "device_replay": device.type == "cuda",
    "fused_optimizer": device.type == "cuda",
    "amp_dtype": None,
    "compile_models": False,
}

VARIANTS = [
    {
        "label": "baseline: linear weighting, constant LR",
        "strategy_weighting": "linear",
        "lr_schedule": "constant",
        "average_replay_reset_minutes": None,
    },
    {
        "label": "uniform weighting, constant LR",
        "strategy_weighting": "uniform",
        "lr_schedule": "constant",
        "average_replay_reset_minutes": None,
    },
    {
        "label": "linear weighting, cosine LR",
        "strategy_weighting": "linear",
        "lr_schedule": "cosine",
        "average_replay_reset_minutes": None,
    },
    {
        "label": "uniform weighting, cosine LR",
        "strategy_weighting": "uniform",
        "lr_schedule": "cosine",
        "average_replay_reset_minutes": None,
    },
    {
        "label": "linear cosine LR, reset average replay every 60m",
        "strategy_weighting": "linear",
        "lr_schedule": "cosine",
        "average_replay_reset_minutes": 60,
    },
]

run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
screen_dir = ARTIFACT_ROOT / f"{spec.to_short_str()}___{run_id}"
screen_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "run_type": "cfr_plus_18_claim_stability_screen",
    "spec": spec.to_json(),
    "run_minutes_per_variant": RUN_MINUTES_PER_VARIANT,
    "eval_target_minutes": EVAL_TARGET_MINUTES,
    "traversals_per_player": TRAVERSALS_PER_PLAYER,
    "seed": SEED,
    "base_trainer_kwargs": {
        **BASE_TRAINER_KWARGS,
        "device": str(device),
    },
    "variants": VARIANTS,
}
(screen_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("screen_dir:", screen_dir)

In [ ]:
def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, default=json_default) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def mean_list(values) -> float:
    if values is None:
        return float("nan")
    if isinstance(values, (int, float)):
        return float(values)
    values = list(values)
    return float(np.mean(values)) if values else float("nan")


def set_optimizer_lr(trainer: DeepCFRPlusTrainer, lr: float) -> None:
    for optimizer in [*trainer.regret_optimizers, *trainer.strategy_optimizers]:
        for group in optimizer.param_groups:
            group["lr"] = float(lr)


def scheduled_lr(variant: dict, measured_s: float, budget_s: float) -> float:
    initial = 1e-3
    final = 1e-4
    if variant["lr_schedule"] == "constant":
        return initial
    progress = min(1.0, max(0.0, measured_s / budget_s))
    return final + 0.5 * (initial - final) * (1.0 + math.cos(math.pi * progress))


def fresh_reservoir_like(buffer, trainer: DeepCFRPlusTrainer):
    if isinstance(buffer, DeviceReservoirBuffer):
        return DeviceReservoirBuffer(
            buffer.capacity,
            buffer.input_dim,
            buffer.action_dim,
            trainer.device,
        )
    if isinstance(buffer, ReservoirBuffer):
        return ReservoirBuffer(buffer.capacity, buffer.input_dim, buffer.action_dim)
    raise TypeError(f"Unsupported strategy buffer type: {type(buffer).__name__}")


def reset_average_replay(trainer: DeepCFRPlusTrainer) -> None:
    # Preserve strategy nets and optimizers. Clear only the memory that defines
    # the average-strategy training distribution.
    trainer.strategy_buffers = [
        fresh_reservoir_like(buffer, trainer)
        for buffer in trainer.strategy_buffers
    ]
    trainer.strategy_validation_buffers = [
        fresh_reservoir_like(buffer, trainer)
        for buffer in trainer.strategy_validation_buffers
    ]


def exact_eval_dense(dense_policy) -> dict:
    start = time.perf_counter()
    _, log = best_response_dense(spec, dense_policy, store_state_values=False)
    exact_br_s = time.perf_counter() - start
    p_first, p_second = log["computer"].exploitability()
    predicted_avg = 0.5 * (p_first + p_second)
    return {
        "p_first": float(p_first),
        "p_second": float(p_second),
        "predicted_avg": float(predicted_avg),
        "exploitability": float(2.0 * predicted_avg - 1.0),
        "exact_br_s": exact_br_s,
    }


def evaluate_trainer(
    trainer: DeepCFRPlusTrainer,
    *,
    variant_label: str,
    variant_slug: str,
    target_min: float,
    measured_s: float,
    variant_dir: Path,
) -> list[dict]:
    rows = []
    snapshot_dir = variant_dir / "snapshots" / f"{int(target_min):04d}m"
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    start = time.perf_counter()
    average_policy = trainer.average_policy()
    save_policy(average_policy, str(snapshot_dir / "learned_average_policy"))
    dense_average = compile_neural_to_dense(average_policy, batch_size=DENSE_COMPILE_BATCH)
    compile_s = time.perf_counter() - start
    row = exact_eval_dense(dense_average)
    row.update({
        "variant": variant_label,
        "variant_slug": variant_slug,
        "policy_kind": "learned_average",
        "target_min": target_min,
        "measured_training_s": measured_s,
        "measured_training_min": measured_s / 60.0,
        "iteration": trainer.iteration,
        "dense_compile_s": compile_s,
        "evaluation_s": compile_s + row["exact_br_s"],
    })
    rows.append(row)

    if device.type == "cuda":
        torch.cuda.empty_cache()

    start = time.perf_counter()
    dense_current = trainer.current_policy_dense(batch_size=DENSE_COMPILE_BATCH)
    compile_s = time.perf_counter() - start
    row = exact_eval_dense(dense_current)
    row.update({
        "variant": variant_label,
        "variant_slug": variant_slug,
        "policy_kind": "current",
        "target_min": target_min,
        "measured_training_s": measured_s,
        "measured_training_min": measured_s / 60.0,
        "iteration": trainer.iteration,
        "dense_compile_s": compile_s,
        "evaluation_s": compile_s + row["exact_br_s"],
    })
    rows.append(row)

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return rows

In [ ]:
budget_s = 60.0 * RUN_MINUTES_PER_VARIANT
eval_targets_s = [60.0 * minute for minute in EVAL_TARGET_MINUTES]

for variant in VARIANTS:
    variant_label = variant["label"]
    variant_slug = slugify(variant_label)
    variant_dir = screen_dir / variant_slug
    training_path = variant_dir / "training.jsonl"
    eval_path = variant_dir / "evaluations.jsonl"
    summary_path = variant_dir / "summary.json"
    final_policy_dir = variant_dir / "final_policy"

    if SKIP_COMPLETED_VARIANTS and summary_path.exists():
        print("skipping completed variant:", variant_label)
        continue

    print("\n===", variant_label, "===")
    variant_dir.mkdir(parents=True, exist_ok=True)
    (variant_dir / "variant.json").write_text(
        json.dumps(variant, indent=2, default=json_default),
        encoding="utf-8",
    )

    trainer_kwargs = deepcopy(BASE_TRAINER_KWARGS)
    trainer_kwargs["strategy_weighting"] = variant["strategy_weighting"]
    trainer = DeepCFRPlusTrainer(spec, **trainer_kwargs)

    measured_s = 0.0
    target_index = 0
    reset_events = []
    reset_minutes = variant.get("average_replay_reset_minutes")
    next_reset_s = None if reset_minutes is None else 60.0 * float(reset_minutes)
    last_print_min = -1

    while measured_s < budget_s:
        lr = scheduled_lr(variant, measured_s, budget_s)
        set_optimizer_lr(trainer, lr)

        start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
        iteration_s = time.perf_counter() - start
        measured_s += iteration_s

        validation = trainer.validation_metrics(max_records=VALIDATION_RECORDS)
        record.update({
            "variant": variant_label,
            "variant_slug": variant_slug,
            "measured_training_s": measured_s,
            "measured_training_min": measured_s / 60.0,
            "iteration_s": iteration_s,
            "learning_rate": lr,
            "validation": validation,
            "mean_regret_loss": mean_list(record.get("regret_loss")),
            "mean_strategy_loss": mean_list(record.get("strategy_loss")),
        })
        append_jsonl(training_path, record)

        if next_reset_s is not None and measured_s >= next_reset_s:
            reset_average_replay(trainer)
            event = {
                "variant": variant_label,
                "measured_training_s": measured_s,
                "measured_training_min": measured_s / 60.0,
                "iteration": trainer.iteration,
                "reset_minutes": reset_minutes,
            }
            reset_events.append(event)
            append_jsonl(variant_dir / "average_replay_resets.jsonl", event)
            print(
                f"[reset average replay] train={measured_s/60:.1f}m "
                f"iter={trainer.iteration}"
            )
            next_reset_s += 60.0 * float(reset_minutes)

        while target_index < len(eval_targets_s) and measured_s >= eval_targets_s[target_index]:
            target_min = EVAL_TARGET_MINUTES[target_index]
            rows = evaluate_trainer(
                trainer,
                variant_label=variant_label,
                variant_slug=variant_slug,
                target_min=target_min,
                measured_s=measured_s,
                variant_dir=variant_dir,
            )
            for row in rows:
                append_jsonl(eval_path, row)
            learned_row = next(row for row in rows if row["policy_kind"] == "learned_average")
            current_row = next(row for row in rows if row["policy_kind"] == "current")
            print(
                f"[eval] target={target_min:>4.0f}m actual={measured_s/60:.1f}m "
                f"iter={trainer.iteration} learned={learned_row['exploitability']:.5f} "
                f"current={current_row['exploitability']:.5f}"
            )
            target_index += 1

        current_min = int(measured_s // 60)
        if current_min != last_print_min and current_min % 10 == 0:
            timing = record["timing"]
            print(
                f"train={measured_s/60:.1f}m iter={trainer.iteration} "
                f"lr={lr:.2e} traverse={timing['traversal_s']:.3f}s "
                f"regret_fit={timing['regret_training_s']:.3f}s "
                f"strategy_fit={timing['strategy_training_s']:.3f}s"
            )
            last_print_min = current_min

    final_policy = trainer.average_policy()
    save_policy(final_policy, str(final_policy_dir))

    training_rows = load_jsonl(training_path)
    evaluation_rows = load_jsonl(eval_path)
    learned_eval = [
        row for row in evaluation_rows
        if row.get("policy_kind") == "learned_average"
    ]
    current_eval = [
        row for row in evaluation_rows
        if row.get("policy_kind") == "current"
    ]

    summary = {
        "variant": variant_label,
        "variant_slug": variant_slug,
        "status": "complete",
        "iterations_completed": trainer.iteration,
        "measured_training_min": measured_s / 60.0,
        "evaluations": len(learned_eval),
        "final_learned_exploitability": learned_eval[-1]["exploitability"] if learned_eval else None,
        "best_learned_exploitability": min((row["exploitability"] for row in learned_eval), default=None),
        "best_learned_min": min(learned_eval, key=lambda row: row["exploitability"])["measured_training_min"] if learned_eval else None,
        "final_current_exploitability": current_eval[-1]["exploitability"] if current_eval else None,
        "best_current_exploitability": min((row["exploitability"] for row in current_eval), default=None),
        "mean_traversal_s": float(np.mean([row["timing"]["traversal_s"] for row in training_rows])),
        "mean_regret_fit_s": float(np.mean([row["timing"]["regret_training_s"] for row in training_rows])),
        "mean_strategy_fit_s": float(np.mean([row["timing"]["strategy_training_s"] for row in training_rows])),
        "reset_events": reset_events,
        "final_policy_dir": str(final_policy_dir),
    }
    summary_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding="utf-8")
    print("completed:", variant_label, "summary:", summary)

In [ ]:
def latest_screen_dir() -> Path:
    candidates = sorted(ARTIFACT_ROOT.glob(f"{spec.to_short_str()}___*"))
    if not candidates:
        raise RuntimeError("No screen output directories found.")
    return candidates[-1]


if "screen_dir" not in globals() or not Path(screen_dir).exists():
    screen_dir = latest_screen_dir()

eval_rows = []
summary_rows = []
for variant_dir in sorted(Path(screen_dir).iterdir()):
    if not variant_dir.is_dir():
        continue
    for row in load_jsonl(variant_dir / "evaluations.jsonl"):
        eval_rows.append(row)
    summary_path = variant_dir / "summary.json"
    if summary_path.exists():
        summary_rows.append(json.loads(summary_path.read_text(encoding="utf-8")))

eval_df = pd.DataFrame(eval_rows)
summary_df = pd.DataFrame(summary_rows)

if not summary_df.empty:
    display(
        summary_df[
            [
                "variant",
                "iterations_completed",
                "measured_training_min",
                "final_learned_exploitability",
                "best_learned_exploitability",
                "best_learned_min",
                "final_current_exploitability",
                "best_current_exploitability",
                "mean_traversal_s",
                "mean_regret_fit_s",
                "mean_strategy_fit_s",
            ]
        ].sort_values("best_learned_exploitability")
    )

if not eval_df.empty:
    display(
        eval_df.pivot_table(
            index=["variant", "policy_kind"],
            values=["exploitability", "dense_compile_s", "exact_br_s", "evaluation_s"],
            aggfunc=["last", "min", "mean"],
        )
    )

In [ ]:
if eval_df.empty:
    raise RuntimeError("No evaluation rows found. Run the training cell first.")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for variant, group in eval_df[eval_df.policy_kind == "learned_average"].groupby("variant"):
    group = group.sort_values("measured_training_min")
    axes[0].plot(
        group["measured_training_min"],
        group["exploitability"],
        marker="o",
        label=variant,
    )
axes[0].set_title("Learned-average exact exploitability")
axes[0].set_xlabel("Measured neural-training minutes")
axes[0].set_ylabel("Exploitability")
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

for variant, group in eval_df[eval_df.policy_kind == "current"].groupby("variant"):
    group = group.sort_values("measured_training_min")
    axes[1].plot(
        group["measured_training_min"],
        group["exploitability"],
        marker="o",
        label=variant,
    )
axes[1].set_title("Current-strategy exact exploitability")
axes[1].set_xlabel("Measured neural-training minutes")
axes[1].set_ylabel("Exploitability")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

learned = eval_df[eval_df.policy_kind == "learned_average"].copy()
for variant, group in learned.groupby("variant"):
    group = group.sort_values("measured_training_min").copy()
    group["best_so_far"] = group["exploitability"].cummin()
    axes[2].plot(
        group["measured_training_min"],
        group["best_so_far"],
        marker="o",
        label=variant,
    )
axes[2].set_title("Learned-average best-so-far")
axes[2].set_xlabel("Measured neural-training minutes")
axes[2].set_ylabel("Best exploitability so far")
axes[2].set_yscale("log")
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=8)

for variant, group in eval_df.groupby("variant"):
    wide = group.pivot_table(
        index="measured_training_min",
        columns="policy_kind",
        values="exploitability",
        aggfunc="last",
    ).reset_index()
    if {"learned_average", "current"}.issubset(wide.columns):
        axes[3].plot(
            wide["measured_training_min"],
            wide["current"] - wide["learned_average"],
            marker="o",
            label=variant,
        )
axes[3].axhline(0.0, color="black", linewidth=1)
axes[3].set_title("Current minus learned-average exploitability")
axes[3].set_xlabel("Measured neural-training minutes")
axes[3].set_ylabel("Current - average")
axes[3].grid(True, alpha=0.3)
axes[3].legend(fontsize=8)

plt.tight_layout()

In [ ]:
train_rows = []
for variant_dir in sorted(Path(screen_dir).iterdir()):
    if not variant_dir.is_dir():
        continue
    for row in load_jsonl(variant_dir / "training.jsonl"):
        train_rows.append(row)

train_df = pd.DataFrame(train_rows)
if train_df.empty:
    raise RuntimeError("No training rows found.")

train_df["traversal_s"] = train_df["timing"].map(lambda item: item["traversal_s"])
train_df["regret_fit_s"] = train_df["timing"].map(lambda item: item["regret_training_s"])
train_df["strategy_fit_s"] = train_df["timing"].map(lambda item: item["strategy_training_s"])
train_df["regret_val_mse"] = train_df["validation"].map(
    lambda item: np.nanmean([entry.get("mse", np.nan) for entry in item["regret"]])
)
train_df["regret_val_tv"] = train_df["validation"].map(
    lambda item: np.nanmean([entry.get("strategy_tv", np.nan) for entry in item["regret"]])
)
train_df["strategy_val_tv"] = train_df["validation"].map(
    lambda item: np.nanmean([entry.get("strategy_tv", np.nan) for entry in item["strategy"]])
)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.ravel()

for variant, group in train_df.groupby("variant"):
    group = group.sort_values("measured_training_min")
    sampled = group.iloc[:: max(1, len(group) // 1000)]
    axes[0].plot(sampled["measured_training_min"], sampled["traversal_s"], label=variant)
    axes[1].plot(sampled["measured_training_min"], sampled["regret_fit_s"], label=variant)
    axes[2].plot(sampled["measured_training_min"], sampled["strategy_fit_s"], label=variant)
    axes[3].plot(sampled["measured_training_min"], sampled["regret_val_mse"], label=variant)
    axes[4].plot(sampled["measured_training_min"], sampled["regret_val_tv"], label=variant)
    axes[5].plot(sampled["measured_training_min"], sampled["strategy_val_tv"], label=variant)

titles = [
    "Traversal seconds",
    "Regret fit seconds",
    "Strategy fit seconds",
    "Regret validation MSE",
    "Regret-induced strategy TV",
    "Average-network validation TV",
]
for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.set_xlabel("Measured neural-training minutes")
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=7)
plt.tight_layout()